# Tahap 4: Pipeline Indexing ke ChromaDB
Notebook ini membangun vector database ChromaDB di folder chroma_db menggunakan embedding BAAI/bge-m3 untuk collection resep dan pedoman_gizi.

In [1]:
import os
import sys
import pandas as pd
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

if sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

torch_lib = os.path.join(sys.prefix, 'Lib', 'site-packages', 'torch', 'lib')
if os.path.exists(torch_lib) and hasattr(os, 'add_dll_directory'):
    try:
        os.add_dll_directory(torch_lib)
    except Exception:
        pass

import torch
if hasattr(torch, 'set_num_threads'):
    try:
        torch.set_num_threads(max(1, (os.cpu_count() or 4) - 1))
    except Exception:
        pass

# Gunakan model BAAI/bge-m3 yang sudah ter-download di cache
local_snapshot = os.path.expanduser(r'~/.cache/huggingface/hub/models--BAAI--bge-m3/snapshots/5617a9f61b028005a4858fdac845db406aefb181')
model_name = local_snapshot if os.path.exists(local_snapshot) else 'BAAI/bge-m3'

print(f"Memuat embedding model: {model_name}")
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

chroma_client = chromadb.PersistentClient(path="chroma_db")
print("ChromaDB PersistentClient berhasil diinisialisasi.")

Memuat embedding model: C:\Users\Bunga Alyssa/.cache/huggingface/hub/models--BAAI--bge-m3/snapshots/5617a9f61b028005a4858fdac845db406aefb181


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

ChromaDB PersistentClient berhasil diinisialisasi.


In [2]:
csv_path = os.path.join("dataset", "cleaned_recipes.csv")
df = pd.read_csv(csv_path)

try:
    chroma_client.delete_collection("resep")
except Exception:
    pass

resep_store = Chroma(
    client=chroma_client,
    collection_name="resep",
    embedding_function=embedding_model
)

recipe_docs = []
recipe_ids = []

for _, row in df.iterrows():
    content = (
        f"Nama Masakan: {row['nama_masakan']}\n"
        f"Kategori: {row['kategori']}\n"
        f"Jumlah Porsi: {row['jumlah_porsi']} ({row['basis_porsi']})\n"
        f"Informasi Nutrisi:\n"
        f"- Kalori: {row['kalori_kkal']} kkal\n"
        f"- Protein: {row['protein_g']} g\n"
        f"- Karbohidrat: {row['karbohidrat_g']} g\n"
        f"- Lemak: {row['lemak_g']} g\n\n"
        f"Bahan-bahan:\n{row['bahan_bahan']}\n\n"
        f"Cara Memasak:\n{row['cara_memasak']}"
    )
    meta = {
        "recipe_id": str(row['recipe_id']),
        "nama_masakan": str(row['nama_masakan']),
        "kategori": str(row['kategori']),
        "jumlah_porsi": int(row['jumlah_porsi']),
        "basis_porsi": str(row['basis_porsi']),
        "kalori_kkal": float(row['kalori_kkal']),
        "protein_g": float(row['protein_g']),
        "karbohidrat_g": float(row['karbohidrat_g']),
        "lemak_g": float(row['lemak_g']),
        "url": str(row['url']) if pd.notna(row['url']) else "",
        "source": "cleaned_recipes.csv"
    }
    doc_id = str(row['recipe_id'])
    recipe_docs.append(Document(page_content=content, metadata=meta))
    recipe_ids.append(doc_id)

batch_size = 100
for i in range(0, len(recipe_docs), batch_size):
    end_idx = min(i + batch_size, len(recipe_docs))
    resep_store.add_documents(documents=recipe_docs[i:end_idx], ids=recipe_ids[i:end_idx])
    print(f"Collection 'resep' terindeks: {end_idx}/{len(recipe_docs)} dokumen")

print(f"Total dokumen collection 'resep': {resep_store._collection.count()}")

Collection 'resep' terindeks: 100/327 dokumen


Collection 'resep' terindeks: 200/327 dokumen


Collection 'resep' terindeks: 300/327 dokumen


Collection 'resep' terindeks: 327/327 dokumen
Total dokumen collection 'resep': 327


In [3]:
try:
    chroma_client.delete_collection("pedoman_gizi")
except Exception:
    pass

pedoman_store = Chroma(
    client=chroma_client,
    collection_name="pedoman_gizi",
    embedding_function=embedding_model
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""]
)

target_files = [
    ("extracted_bab2_pedoman_gizi.md", "Skripsi UII - Bab 2.1 dan Lampiran D"),
    ("extracted_permenkes_akg_2019.md", "Permenkes No. 28/2019 - AKG"),
    ("extracted_permenkes_gizi_seimbang_2014.md", "Permenkes No. 41/2014 - Pedoman Gizi Seimbang")
]

gizi_docs = []
gizi_ids = []
counter = 1

for fname, desc in target_files:
    fpath = os.path.join("doc", fname)
    if not os.path.exists(fpath):
        continue
    with open(fpath, "r", encoding="utf-8") as f:
        text = f.read()
    chunks = splitter.split_text(text)
    print(f"File {fname}: {len(chunks)} chunk dibuat")
    for c in chunks:
        cid = f"GIZI_{counter:04d}"
        meta = {
            "chunk_id": cid,
            "file_name": fname,
            "source": desc,
            "type": "pedoman_gizi"
        }
        gizi_docs.append(Document(page_content=c, metadata=meta))
        gizi_ids.append(cid)
        counter += 1

for i in range(0, len(gizi_docs), batch_size):
    end_idx = min(i + batch_size, len(gizi_docs))
    pedoman_store.add_documents(documents=gizi_docs[i:end_idx], ids=gizi_ids[i:end_idx])
    print(f"Collection 'pedoman_gizi' terindeks: {end_idx}/{len(gizi_docs)} chunk")

print(f"Total chunk collection 'pedoman_gizi': {pedoman_store._collection.count()}")

File extracted_bab2_pedoman_gizi.md: 38 chunk dibuat


File extracted_permenkes_akg_2019.md: 38 chunk dibuat
File extracted_permenkes_gizi_seimbang_2014.md: 56 chunk dibuat


Collection 'pedoman_gizi' terindeks: 100/132 chunk


Collection 'pedoman_gizi' terindeks: 132/132 chunk
Total chunk collection 'pedoman_gizi': 132
